In [ ]:
# @title Cell 1: Environment Setup
# @markdown **Compute Requirements:** A standard CPU is perfectly sufficient for this 2-layer demo. If running in Colab, the free T4 GPU will execute the training loop near-instantly.
# @markdown
# @markdown Run this cell to install `transformer_lens`. You may need to restart the runtime after installation.
%pip install -q transformer_lens

In [ ]:
# @title Cell 2: Imports & Model Configuration
# @markdown Initializes a tiny 2-layer model optimized for the induction head task.
import torch
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import collections
from transformer_lens import HookedTransformer, HookedTransformerConfig

# Configuration
torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Running on {device}")

cfg = HookedTransformerConfig(
    n_layers=2,
    d_model=64,
    d_head=32,
    n_heads=2,
    d_mlp=256,
    d_vocab=64,
    n_ctx=32,
    act_fn="gelu",
    normalization_type="LN",
    seed=42,
)
model = HookedTransformer(cfg).to(device)

In [ ]:
# @title Cell 3: The AttentionTelemetry Bridge
# @markdown A lightweight, zero-dependency bridge to extract mechanistic metrics
# @markdown (Coherence and Agreement) directly from an `ActivationCache`.
class AttentionTelemetry:
    """Lightweight bridge extracting mechanistic metrics from ActivationCache."""

    @staticmethod
    def compute_metrics(cache, layer_idx):
        """Computes attention coherence and agreement for a given layer.

        Args:
            cache (ActivationCache): The cached activations from the forward pass.
            layer_idx (int): The index of the layer to analyze.

        Returns:
            dict: A dictionary containing layer_idx, head_coherence, and head_agreement.

        Notes on v_max (0.005):
            The agreement normalization constant is derived from the expected inter-head
            attention variance at convergence in 2-layer induction head toy models.
            At convergence, heads specialize (low variance); pre-convergence variance
            peaks near 0.005. This value is task- and architecture-specific; adjust
            if adapting to larger models or different tasks.
        """

        pattern_name = f"blocks.{layer_idx}.attn.hook_pattern"

        # Shape: [batch, heads, seq, seq]
        attn_probs = cache[pattern_name]

        # 1. Head Coherence: 1.0 - (Entropy / Max_Entropy)
        probs = attn_probs + 1e-9
        entropy = -torch.sum(probs * torch.log(probs), dim=-1)  # [batch, heads, seq]
        head_coherence = 1.0 - (entropy.mean(dim=[0, 2]) / np.log(attn_probs.shape[-1]))

        # 2. Head Agreement: 1.0 - clip(Variance / v_max)
        mean_var = torch.var(attn_probs, dim=1).mean()  # Variance across heads
        head_agreement = 1.0 - torch.clamp(mean_var / 0.005, 0.0, 1.0)

        return {
            "layer_idx": layer_idx,
            "head_coherence": head_coherence.mean().item(),
            "head_agreement": head_agreement.item()
        }

In [ ]:
# @title Cell 4: Synthetic Data Generator (Induction Task)
# @markdown Generates `[A, B, C, ..., A, B, C]` sequences to force the model to learn in-context look-back circuits.
def generate_induction_data(batch_size, seq_len, vocab_size, device="cpu"):
    half_len = seq_len // 2
    first_half = torch.randint(0, vocab_size, (batch_size, half_len))
    data = torch.cat([first_half, first_half], dim=1)
    return data.to(device)

In [ ]:
## @title Cell 5: Training Loop with Real-Time Telemetry
# @markdown Trains the model and captures metrics every 10 steps using dynamic dictionary logging.
# @markdown `run_with_cache` is only called at log steps to avoid 10x unnecessary overhead.
steps = 500
log_interval = 10
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

# Dynamic logging scales automatically regardless of n_layers
telemetry = collections.defaultdict(list)

print(f"📊 Starting Training Loop ({steps} steps)...")

for step in range(steps):
    model.train()
    optimizer.zero_grad()

    tokens = generate_induction_data(64, 32, 64, device)

    if step % log_interval == 0:
        logits, cache = model.run_with_cache(tokens[:, :-1])
        telemetry["step"].append(step)
        telemetry["loss"].append(
            F.cross_entropy(logits.reshape(-1, logits.size(-1)), tokens[:, 1:].reshape(-1)).item()
        )
        # Automatically creates keys for any number of layers
        for i in range(model.cfg.n_layers):
            metrics = AttentionTelemetry.compute_metrics(cache, i)
            telemetry[f"layer_{i}_coherence"].append(metrics["head_coherence"])
            telemetry[f"layer_{i}_agreement"].append(metrics["head_agreement"])
    else:
        logits = model(tokens[:, :-1])

    loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), tokens[:, 1:].reshape(-1))
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print(f"   Step {step:03d}: Loss = {loss.item():.4f}")

print("✅ Training & Telemetry Capture Complete.")

In [ ]:
# @title Cell 6: Visualization (Phase Transition & Heatmap)
# @markdown Renders the final publication-ready dual plot directly from the telemetry dictionary.
plt.style.use('default')
fig = plt.figure(figsize=(12, 10))

# --- Plot A: Training Dynamics (Loss vs Coherence) ---
ax1 = fig.add_subplot(2, 1, 1)

last_layer_idx = model.cfg.n_layers - 1  # FIX: derive label dynamically

color_loss = 'tab:gray'
ax1.set_xlabel('Training Step', fontweight='bold')
ax1.set_ylabel('Loss (Cross Entropy)', color=color_loss, fontweight='bold')
ax1.plot(telemetry['step'], telemetry['loss'], color=color_loss, linestyle='--', alpha=0.8, label='Loss')
ax1.tick_params(axis='y', labelcolor=color_loss)
ax1.set_title('Phase Transition: Loss Crash vs. Circuit Formation', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
color_coh = '#1f77b4'
# FIX: y-axis label was hardcoded 'Layer 1 Head Coherence' regardless of n_layers
ax2.set_ylabel(f'Layer {last_layer_idx} Head Coherence', color=color_coh, fontweight='bold')
ax2.plot(
    telemetry['step'],
    telemetry[f'layer_{last_layer_idx}_coherence'],
    color=color_coh, linewidth=2.5,
    label=f'Layer {last_layer_idx} Coherence'
)
ax2.tick_params(axis='y', labelcolor=color_coh)

# --- Plot B: The Telemetry Heatmap ---
ax3 = fig.add_subplot(2, 1, 2)
num_layers = model.cfg.n_layers
steps_array = telemetry['step']

# Construct Heatmap Matrix directly from the dictionary
heatmap_data = np.zeros((num_layers, len(steps_array)))
for layer in range(num_layers):
    heatmap_data[layer, :] = telemetry[f"layer_{layer}_coherence"]

im = ax3.imshow(
    heatmap_data,
    aspect='auto',
    cmap='magma',
    interpolation='nearest',
    extent=[steps_array[0], steps_array[-1], -0.5, num_layers - 0.5],
    origin='lower'
)

ax3.set_title('Attention Coherence Heatmap by Layer Depth', fontsize=14, fontweight='bold')
ax3.set_ylabel('Model Depth (Layer Index)', fontweight='bold')
ax3.set_xlabel('Training Step', fontweight='bold')
ax3.set_yticks(range(num_layers))
ax3.set_yticklabels([f"Layer {i}" for i in range(num_layers)])

cbar = plt.colorbar(im, ax=ax3, orientation='horizontal', pad=0.15)
cbar.set_label('Head Coherence (0.0 to 1.0)', fontweight='bold')

plt.tight_layout()
plt.show()